# **Sensibilidad global de $\mathcal{R}_0$ con `pyR0compute`**

El índice de sensibilidad normalizado
$$\Upsilon_p = \frac{\partial \mathcal{R}_0}{\partial p}\,\frac{p}{\mathcal{R}_0}$$
es **local**: se evalúa en un punto del espacio de parámetros y no recoge ni la incertidumbre de los parámetros ni sus interacciones. Este cuaderno muestra los dos métodos **globales** que incorpora la librería:

* **LHS-PRCC** (Marino et al., 2008): muestreo de hipercubo latino y coeficientes de correlación parcial de rangos. Mide la fuerza y el signo de una relación **monótona** entre cada parámetro y $\mathcal{R}_0$.
* **Índices de Sobol** (Saltelli et al., 2010): descomposición de la varianza de $\mathcal{R}_0$. El índice de primer orden $S_1$ es la fracción de $\mathrm{Var}(\mathcal{R}_0)$ explicada por un parámetro solo; el total $S_T$ suma además todas sus interacciones.

Como $\mathcal{R}_0$ está en forma cerrada, la librería lo evalúa vectorizado con NumPy, de modo que decenas de miles de evaluaciones toman segundos.

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yvan-BM-Git/pyR0compute/blob/main/examples/02_sensibilidad_global.ipynb)

In [ ]:
# Instala pyR0compute desde PyPI con el extra [global] (SciPy >= 1.11).
# La versión en desarrollo: %pip install "pyR0compute[global] @ git+https://github.com/Yvan-BM-Git/pyR0compute"
%pip install "pyR0compute[global]>=0.2"

In [ ]:
import numpy as np                            # Cálculo numérico
import sympy as sp                            # Cálculo simbólico
import matplotlib.pyplot as plt               # Gráficos
from pyr0compute import R0Model               # Clase principal de la librería
from IPython.display import Markdown, display # Para mostrar tablas en Markdown

## 1. Modelo SEIR con dinámica vital

$$
\begin{cases}
\dfrac{dS}{dt} = \Lambda - \beta SI - \mu S \\[6pt]
\dfrac{dE}{dt} = \beta SI - (\sigma + \mu)E \\[6pt]
\dfrac{dI}{dt} = \sigma E - (\gamma + \mu)I \\[6pt]
\dfrac{dR}{dt} = \gamma I - \mu R
\end{cases}
\qquad
\mathcal{R}_0 = \frac{\Lambda\beta\sigma}{\mu(\gamma+\mu)(\sigma+\mu)}
$$

In [ ]:
seir = R0Model('''
    dS/dt = Lambda - beta*S*I - mu*S
    dE/dt = beta*S*I - (sigma + mu)*E
    dI/dt = sigma*E - (gamma + mu)*I
    dR/dt = gamma*I - mu*R
''', infected=["E", "I"])

base = {"Lambda": 10, "beta": 0.002, "mu": 0.02, "sigma": 0.2, "gamma": 0.1}   # valores de referencia
print("R0 =", seir.R0)
print("R0 en el punto de referencia:", round(seir.R0_numeric(base), 4))

### 1.1 Punto de partida: índices locales

Los índices locales en el punto de referencia sirven para comparar con los globales.

In [ ]:
local = seir.sensitivity_indices(base)                 # índices normalizados en el punto base
for p, v in sorted(local.items(), key=lambda kv: -abs(kv[1])):
    print(f"{str(p):>7s}: {v:+.4f}")

### 1.2 LHS-PRCC

Con `baseline` y `spread=0.3`, cada parámetro varía uniformemente en $\pm 30\,\%$ de su valor de referencia. La librería comprueba además, con el signo de $\partial\mathcal{R}_0/\partial p$ en cada muestra, que $\mathcal{R}_0$ sea monótona en cada parámetro (columna `monotonic`), condición necesaria para interpretar el PRCC.

In [ ]:
res_prcc = seir.prcc(baseline=base, spread=0.3, n=2000, seed=1)
res_prcc

In [ ]:
res_prcc.to_dataframe()           # tabla ordenada por |PRCC| (requiere pandas)

In [ ]:
res_prcc.plot();                  # * marca p < 0.05

Las muestras quedan guardadas en `samples` y `output`, lo que permite inspeccionar visualmente la monotonía:

In [ ]:
names = [p.name for p in res_prcc.parameters]
fig, axes = plt.subplots(1, len(names), figsize=(3 * len(names), 2.8), sharey=True)
for ax, j, name in zip(axes, range(len(names)), names):
    ax.scatter(res_prcc.samples[:, j], res_prcc.output, s=3, alpha=0.3, color="#2a78d6")
    ax.set_xlabel(f"${sp.latex(res_prcc.parameters[j])}$")
axes[0].set_ylabel(r"$\mathcal{R}_0$")
fig.tight_layout()

La tabla en LaTeX (o en Markdown, para el cuaderno) está lista para pegar en un artículo:

In [ ]:
display(Markdown(res_prcc.to_latex("markdown")))
print(res_prcc.to_latex())        # versión para el documento (\begin{table} ... \end{table})

### 1.3 Índices de Sobol

Se usan las mismas distribuciones. El costo es $n(k+2)$ evaluaciones de $\mathcal{R}_0$, con $n$ potencia de 2 y $k$ parámetros que varían. Los intervalos son bootstrap al 95 %.

In [ ]:
res_sobol = seir.sobol_indices(baseline=base, spread=0.3, n=2**12, seed=1)
res_sobol

In [ ]:
res_sobol.plot();

$\sum_i S_{1,i} < 1$ y $S_T > S_1$ indican interacciones: $\mathcal{R}_0$ es un cociente de productos, de modo que el efecto de un parámetro depende del valor de los demás. La diferencia crece con la amplitud de los rangos:

In [ ]:
for spread in (0.1, 0.3, 0.6, 0.9):
    r = seir.sobol_indices(baseline=base, spread=spread, n=2**12, seed=1, confidence_level=None)
    print(f"±{spread:.0%}:  sum(S1) = {r.indices['S1'].sum():.3f},  "
          f"max(ST - S1) = {(r.indices['ST'] - r.indices['S1']).max():.3f}")

### 1.4 Distribuciones a medida

Además de `(low, high)` (uniforme) se aceptan `("loguniform", low, high)`, `("normal", media, de)`, `("truncnormal", media, de, low, high)`, `("triangular", low, moda, high)` o cualquier distribución congelada de `scipy.stats`. Los parámetros de `fixed` se mantienen constantes.

In [ ]:
dists = {
    "beta":  ("loguniform", 1e-3, 4e-3),                 # varios órdenes de magnitud
    "mu":    ("truncnormal", 0.02, 0.005, 0.005, 0.05),  # normal truncada para que mu > 0
    "gamma": (0.05, 0.2),                                # uniforme
}
seir.sobol_indices(dists, fixed={"Lambda": 10, "sigma": 0.2}, n=2**12, seed=1)

## 2. Del índice local al global: Ross-Macdonald

$$
\begin{cases}
\dfrac{dS_h}{dt} = \Lambda_h - a b_h \dfrac{S_h I_v}{N_h} - \mu_h S_h \\[6pt]
\dfrac{dI_h}{dt} = a b_h \dfrac{S_h I_v}{N_h} - (\gamma + \mu_h) I_h \\[6pt]
\dfrac{dR_h}{dt} = \gamma I_h - \mu_h R_h \\[6pt]
\dfrac{dS_v}{dt} = \Lambda_v - a b_v \dfrac{S_v I_h}{N_h} - \mu_v S_v \\[6pt]
\dfrac{dI_v}{dt} = a b_v \dfrac{S_v I_h}{N_h} - \mu_v I_v
\end{cases}
$$

Con $\gamma$ y $\mu_h$ fijos, $\mathcal{R}_0 = c\prod_i p_i^{a_i}$ es un monomio. Entonces $\Upsilon_{p_i} = a_i$ en todo punto y $\log\mathcal{R}_0 = \log c + \sum_i a_i \log p_i$ es aditivo. Si los parámetros son log-uniformes e independientes, los índices de Sobol de $\log\mathcal{R}_0$ son exactamente
$$
S_{1,i} = S_{T,i} = \frac{a_i^2\,\mathrm{Var}(\log p_i)}{\sum_j a_j^2\,\mathrm{Var}(\log p_j)},
\qquad \mathrm{Var}(\log p_i) = \frac{\left(\log(b_i/a'_i)\right)^2}{12}
\ \text{para}\ p_i \sim \mathrm{LogU}(a'_i, b_i).
$$
El índice global combina la elasticidad local $a_i$ con la incertidumbre del parámetro.

In [ ]:
rm = R0Model('''
    dS_h/dt = Lambda_h - a*b_h*S_h*I_v/N_h - mu_h*S_h
    dI_h/dt = a*b_h*S_h*I_v/N_h - (gamma + mu_h)*I_h
    dR_h/dt = gamma*I_h - mu_h*R_h
    dS_v/dt = Lambda_v - a*b_v*S_v*I_h/N_h - mu_v*S_v
    dI_v/dt = a*b_v*S_v*I_h/N_h - mu_v*I_v
''', infected=["I_h", "I_v"])
rm.R0

In [ ]:
ranges = {"a": (0.1, 0.5), "b_h": (0.1, 0.9), "b_v": (0.1, 0.9), "Lambda_h": (50, 200),
          "Lambda_v": (1000, 10000), "mu_v": (0.05, 0.2), "N_h": (2000, 8000)}
fixed = {"gamma": 0.1, "mu_h": 0.02}

# Elasticidades locales (constantes: no dependen del punto)
point = {**{p: np.sqrt(lo * hi) for p, (lo, hi) in ranges.items()}, **fixed}
upsilon = {str(p): v for p, v in rm.sensitivity_indices(point).items()}

# Índices de Sobol de log R0 y su valor analítico
res_rm = rm.sobol_indices({p: ("loguniform",) + r for p, r in ranges.items()},
                          fixed=fixed, n=2**13, seed=2, log_output=True)
w = {p: upsilon[p]**2 * np.log(hi / lo)**2 / 12 for p, (lo, hi) in ranges.items()}
total = sum(w.values())

rows = ["| $p$ | $\\Upsilon_p$ | $S_1$ | $S_T$ | analítico |", "|---|---:|---:|---:|---:|"]
for p in res_rm.ranking():
    r = res_rm[p]
    rows.append(f"| ${sp.latex(sp.Symbol(p))}$ | {upsilon[p]:+.2f} | {r['S1']:.3f} | {r['ST']:.3f} | {w[p]/total:.3f} |")
display(Markdown("\n".join(rows)))

Los exponentes $1/2$ de $b_h, b_v, \Lambda_h, \Lambda_v$ reflejan la convención de la raíz cuadrada (radio espectral de $K = FV^{-1}$). Con la otra convención ($\mathcal{R}_0^2$, el número de casos en dos generaciones) todas las elasticidades se duplican, pero los índices de Sobol de $\log\mathcal{R}_0$ no cambian, porque son razones de varianzas.

## 3. Cuando el PRCC falla: dependencia no monótona

Si la transmisibilidad depende de un factor $a(1-a)$, con $a \in (0, 1)$,
$$
\frac{dS}{dt} = \Lambda - \beta a(1-a) SI - \mu S, \qquad
\frac{dI}{dt} = \beta a(1-a) SI - (\gamma+\mu) I,
\qquad
\mathcal{R}_0 = \frac{\Lambda\beta a(1-a)}{\mu(\gamma+\mu)},
$$
$\mathcal{R}_0$ crece y luego decrece en $a$. El PRCC de $a$ es cercano a 0 aunque $a$ domine la variabilidad de $\mathcal{R}_0$; la librería lo detecta con la derivada simbólica y emite una advertencia. Los índices de Sobol, que no suponen monotonía, sí identifican a $a$.

In [ ]:
nm = R0Model('''
    dS/dt = Lambda - beta*a*(1 - a)*S*I - mu*S
    dI/dt = beta*a*(1 - a)*S*I - (gamma + mu)*I
''', infected=["I"])

dists_nm = {"a": (0.01, 0.99), "beta": (1, 2)}
fixed_nm = {"Lambda": 1, "mu": 0.1, "gamma": 0.2}
nm.prcc(dists_nm, fixed=fixed_nm, n=1000, seed=0)

In [ ]:
nm.sobol_indices(dists_nm, fixed=fixed_nm, n=2**12, seed=0)

## Recomendaciones de uso

* Reporte el PRCC solo para parámetros con `monotonic = True`; en caso contrario use Sobol.
* Para los índices de Sobol, aumente $n$ hasta que los intervalos de confianza sean estrechos; valores levemente negativos de $S_1$ en parámetros poco influyentes son ruido de Monte Carlo.
* Ambos métodos suponen parámetros independientes; si hay restricciones entre ellos (por ejemplo $N = \Lambda/\mu$), parametrice el modelo de forma que las cantidades muestreadas sean independientes.
* Fije `seed` para que los resultados sean reproducibles.

## Referencias

* Marino, S., Hogue, I. B., Ray, C. J., & Kirschner, D. E. (2008). A methodology for performing global uncertainty and sensitivity analysis in systems biology. *Journal of Theoretical Biology*, 254(1), 178-196.
* Saltelli, A., Annoni, P., Azzini, I., Campolongo, F., Ratto, M., & Tarantola, S. (2010). Variance based sensitivity analysis of model output. Design and estimator for the total sensitivity index. *Computer Physics Communications*, 181(2), 259-270.
* Sobol', I. M. (2001). Global sensitivity indices for nonlinear mathematical models and their Monte Carlo estimates. *Mathematics and Computers in Simulation*, 55(1-3), 271-280.
* Chitnis, N., Hyman, J. M., & Cushing, J. M. (2008). Determining important parameters in the spread of malaria through the sensitivity analysis of a mathematical model. *Bulletin of Mathematical Biology*, 70(5), 1272-1296.